# Phase Retrieval Core With Gradient Refinement

This notebook shows the integrated `phase_retrieval_core` workflow where gradient descent is just another recipe stage. The stage uses the same Fourier-field and centered support-mask conventions as the normal ER/HAPRE/HIO stages.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from skimage.draw import disk

# Make imports work when Jupyter starts in either the repository root or notebooks/.
REPO_ROOT = Path.cwd() if (Path.cwd() / "library").is_dir() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from library import phase_retrieval_core as pr
from library import phase_retrieval_gradient as gd

plt.rcParams.update({"figure.figsize": (7, 5), "image.cmap": "gray"})


## Build A Small Synthetic Hologram

The synthetic object is defined in the centered display frame. `gd.display_object_to_fourier_field()` converts it to the Fourier-field convention returned by the phase-retrieval libraries.


In [ ]:
shape = (64, 64)
yy_grid, xx_grid = np.indices(shape)

supportmask = np.zeros(shape, dtype=float)
yy, xx = disk((shape[0] // 2, shape[1] // 2), 12, shape=shape)
supportmask[yy, xx] = 1.0

amplitude = supportmask * (0.5 + 0.25 * np.cos((yy_grid - shape[1] / 2) / 1.5)**2* np.cos((xx_grid - shape[1] / 2) / 1.0))
phase = supportmask * (0.7 * np.cos((xx_grid +yy_grid - shape[0] / 2) / 2.0) * np.sin((xx_grid-yy_grid - shape[0] / 2) / 5.0))
true_object = amplitude * np.exp(1j * phase)

true_field = gd.display_object_to_fourier_field(true_object)
pos_hologram = np.abs(true_field) ** 2
neg_hologram = pos_hologram.copy()

mask_pixel = np.zeros(shape, dtype=int)
yy, xx = disk((shape[0] // 2, shape[1] // 2), 2, shape=shape)
mask_pixel[yy, xx] = 1

target_amplitude = np.sqrt(pos_hologram)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(np.abs(true_object))
axes[0].set_title("true object abs")
axes[1].imshow(np.angle(true_object), cmap="twilight")
axes[1].set_title("true object phase")
axes[2].imshow(np.log1p(pos_hologram))
axes[2].set_title("synthetic hologram")
for axis in axes:
    axis.axis("off")
plt.tight_layout()


## Run One Recipe With Full, Partial, And Gradient Stages

The core recipe can mix ordinary full-coherence stages, Richardson-Lucy partial-coherence stages, and `"gradient_descent"` stages. For `"gradient_descent"`, the recipe keys have these meanings:

- `number_iterations`: number of gradient steps;
- `beta_zero` and `beta_mode`: learning-rate schedule;
- `alpha_zero` and `alpha_mode`: support-loss weight schedule;
- `Fourier_last`: optionally reapply measured Fourier amplitudes after the gradient stage.

The high-level return tuple is unchanged for backward compatibility. To inspect the result of each stage type, use `errors["latest"]["full_coherence"]`, `errors["latest"]["partial_coherence"]`, and `errors["latest"]["gradient_descent"]`. Every step also stores `field_after`.


In [ ]:
recipe = {
    "algorithm_list": ["OSS", "OSS", "ER", "gradient_descent"],
    "number_iterations": [1, 1, 1, 15000],
    "helicity": ["pos", "pos", "pos", "pos"],
    "beta_zero": [0.5, 0.5, 0.5, 1300.0],       # 500.0 is the gradient learning rate
    "beta_mode": ["arctan", "const", "const", "const"],
    "alpha_zero": [0.0, 0.0, 0.0, 0.81],        # 1.0 is the gradient support weight
    "alpha_mode": ["const", "const", "const", "const"],
    "RL_its": [0, 0, 0, 0],                   # third stage enables partial coherence
    "RL_freqs": [1e9, 1e9, 1e9, 1e9],
    "TV_freqs": [1e9, 1e9, 1e9, 1e9],
    "plot_every": [10, 5, 1, 1],
    "average_img": [3, 3, 1, 1],
    "Fourier_last": [False, False, False, False],
    "output": [False, False, True, True],
    "hologram_intensity_cutoff_vmin": -1,
    "Startimage": [None, "pos", "pos", "pos"],
    "Startgamma": [None, None, "pos", None],
}

results = pr.phase_retrieval_algorithm(
    pos_hologram,
    neg_hologram,
    mask_pixel,
    supportmask,
    phase_retrieval_recipe=recipe,
)

(
    retrieved_pos,
    retrieved_neg,
    retrieved_pos_partial,
    retrieved_neg_partial,
    bsmask_pos,
    bsmask_neg,
    gamma_pos,
    gamma_neg,
    errors,
) = results

full_field = errors["latest"]["full_coherence"]["pos"]
partial_field = errors["latest"]["partial_coherence"]["pos"]
gradient_field = errors["latest"]["gradient_descent"]["pos"]
selected_outputs = errors["outputs"]

gradient_step = next(
    step for step in reversed(errors["steps"])
    if step["mode"] == "gradient_descent"
)
if partial_field==None:
    partial_field=full_field*0



## Compare The Three Stage Outputs

Use `gd.fourier_field_to_display_object()` for normal centered plotting. The internal support-space object is shifted relative to this display frame by design.


In [ ]:
object_full = gd.fourier_field_to_display_object(full_field)
object_partial = gd.fourier_field_to_display_object(partial_field)
object_gradient = gd.fourier_field_to_display_object(gradient_field)

full_diffraction_loss = gd.diffraction_loss(full_field, target_amplitude, mask_pixel)
full_support_loss = gd.support_loss(full_field, supportmask)
partial_diffraction_loss = gd.diffraction_loss(partial_field, target_amplitude, mask_pixel)
partial_support_loss = gd.support_loss(partial_field, supportmask)
gradient_diffraction_loss = gd.diffraction_loss(gradient_field, target_amplitude, mask_pixel)
gradient_support_loss = gd.support_loss(gradient_field, supportmask)

print("full diffraction/support:", full_diffraction_loss, full_support_loss)
print("partial diffraction/support:", partial_diffraction_loss, partial_support_loss)
print("gradient diffraction/support:", gradient_diffraction_loss, gradient_support_loss)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 6))

axes[0, 0].imshow(np.abs(object_full))
axes[0, 0].set_title("full coherence abs")
axes[0, 1].imshow(np.abs(object_partial))
axes[0, 1].set_title("partial coherence abs")
axes[0, 2].plot(gradient_step["error"], label="diffraction")
axes[0, 2].plot(gradient_step["support_error"], label="support")
axes[0, 2].set_yscale("log")
axes[0, 2].set_title("gradient losses")
axes[0, 2].legend()

axes[1, 0].imshow(np.abs(object_gradient))
axes[1, 0].set_title("gradient abs")
axes[1, 1].imshow(np.angle(object_gradient), cmap="twilight")
axes[1, 1].set_title("gradient phase")
axes[1, 2].imshow(np.abs(object_gradient) * (1 - supportmask))
axes[1, 2].set_title("gradient outside support")

for axis in axes.ravel():
    if not axis.lines:
        axis.axis("off")
plt.tight_layout()


## Tuning Notes

If the gradient losses oscillate or rise, reduce the gradient-stage `beta_zero` or use a decreasing `beta_mode`, for example `"linear_to_0"`. Increase `alpha_zero` when the reconstruction leaks outside the support, and decrease it when support regularization overwhelms the Fourier-amplitude fit.

`Fourier_last=True` on the gradient stage is useful when the final output must exactly satisfy the measured amplitudes on valid detector pixels. `Fourier_last=False` is useful when comparing the raw gradient objective.
